# Bike Trip Data — Pandas Practice Notebook
## CodeSignal Prep: Timestamps · cut/qcut · melt · rolling · cumsum · expanding · pipe

**How to use:**
- Read the problem in each section
- Write your solution in the `# YOUR CODE HERE` cell
- Run the ✅ CHECK cell to verify
- Reveal the 💡 SOLUTION cell only if stuck

**Dataset:** Divvy / Citi Bike new schema — the most likely format in your test.

In [ ]:
import pandas as pd
import numpy as np

# ── DATASET ───────────────────────────────────────────────────────────────────
raw = pd.DataFrame({
    "ride_id": [f"R{str(i).zfill(3)}" for i in range(1, 31)],
    "rideable_type": [
        "classic_bike","electric_bike","classic_bike","docked_bike","electric_bike",
        "classic_bike","electric_bike","classic_bike","classic_bike","electric_bike",
        "classic_bike","docked_bike","classic_bike","electric_bike","classic_bike",
        "classic_bike","electric_bike","classic_bike","classic_bike","docked_bike",
        "electric_bike","classic_bike","classic_bike","electric_bike","classic_bike",
        "classic_bike","electric_bike","classic_bike","docked_bike","classic_bike",
    ],
    "started_at": pd.to_datetime([
        "2024-03-01 07:05","2024-03-01 08:47","2024-03-01 12:15","2024-03-01 17:32","2024-03-01 22:10",
        "2024-03-02 07:58","2024-03-02 09:01","2024-03-02 13:22","2024-03-02 18:05","2024-03-03 08:12",
        "2024-03-03 08:50","2024-03-03 19:40","2024-03-04 07:15","2024-03-04 09:20","2024-03-04 12:00",
        "2024-03-05 07:30","2024-03-05 11:00","2024-03-05 17:45","2024-03-06 08:05","2024-03-06 14:30",
        "2024-03-07 09:00","2024-03-07 17:00","2024-03-08 07:00","2024-03-08 10:15","2024-03-08 13:00",
        "2024-03-09 08:30","2024-03-09 12:00","2024-03-10 07:10","2024-03-10 16:00","2024-03-10 19:30",
    ]),
    "ended_at": pd.to_datetime([
        "2024-03-01 07:25","2024-03-01 09:32","2024-03-01 12:55","2024-03-01 17:55","2024-03-01 22:48",
        "2024-03-02 08:30","2024-03-02 09:46","2024-03-02 14:10","2024-03-02 18:35","2024-03-03 08:42",
        "2024-03-03 10:15","2024-03-03 20:00","2024-03-04 07:45","2024-03-04 10:05","2024-03-04 12:20",
        "2024-03-05 08:00","2024-03-05 11:55","2024-03-05 18:10","2024-03-06 08:35","2024-03-06 15:20",
        "2024-03-07 09:50","2024-03-07 17:30","2024-03-08 07:25","2024-03-08 11:10","2024-03-08 13:22",
        "2024-03-09 09:00","2024-03-09 12:45","2024-03-10 07:40","2024-03-10 16:45","2024-03-10 20:00",
    ]),
    "start_station_id": [
        "13022","13008","KA135","KA123","13022","KA135","TA039","13008","13022","KA135",
        "KA123","13008","13022","13008","TA029","KA135","13022","13008","WL008","TA039",
        "13022","13008","KA123","13300","13022","13008","TA029","KA135","13022","13008",
    ],
    "end_station_id": [
        "13008","KA135","13022","13008","KA135","13022","13008","KA123","KA135","13022",
        "13022","KA135","TA039","WL008","13022","13008","TA029","13022","KA123","13300",
        "13008","13022","13008","TA029","TA039","KA135","WL008","KA123","TA029","13022",
    ],
    "member_casual": [
        "member","casual","member","member","casual","member","casual","member","casual","member",
        "member","casual","member","casual","member","member","casual","member","member","casual",
        "casual","member","member","casual","member","member","casual","member","casual","member",
    ],
})

stations = pd.DataFrame({
    "station_id":   ["13022","13008","KA135","KA123","TA039","TA029","13300","WL008"],
    "station_name": ["Streeter Dr & Grand Ave","Millennium Park","Clark St & Elm St",
                     "Kingsbury St & Kinzie St","Wabash Ave & Grand Ave","Lake Shore Dr & Monroe",
                     "DuSable Lake Shore Dr","Clinton St & Madison St"],
    "region":       ["Lakefront","Loop","Near North","River North","Loop","Lakefront","Lakefront","West Loop"],
    "capacity":     [23, 35, 15, 19, 27, 31, 20, 18],
})

print(f"Trips: {raw.shape}  |  Stations: {stations.shape}")
raw.head(3)

# 🕐 SECTION 1 — TIMESTAMP TRANSFORMATIONS

## P1 — Basic timestamp features
Add these columns to `df`:
- `duration_sec` — trip duration in seconds (int)
- `duration_min` — duration in minutes (float, 2 dp)
- `hour_of_day` — hour trip started (0–23)
- `day_of_week` — name e.g. "Monday"
- `is_weekend` — True if Saturday or Sunday (bool)
- `date` — date only portion of started_at
- `start_hour` — started_at floored to the nearest hour

In [ ]:
df = raw.copy()

# YOUR CODE HERE


df.head(3)

In [ ]:
# ✅ CHECK P1
assert df.loc[0, "duration_sec"] == 1200, "duration_sec wrong"
assert df.loc[0, "duration_min"] == 20.0, "duration_min wrong"
assert df.loc[0, "hour_of_day"] == 7, "hour_of_day wrong"
assert df.loc[0, "day_of_week"] == "Friday", "day_of_week wrong"
assert df.loc[0, "is_weekend"] == False, "is_weekend wrong"
assert str(df.loc[0, "date"]) == "2024-03-01", "date wrong"
assert df.loc[0, "start_hour"] == pd.Timestamp("2024-03-01 07:00"), "start_hour wrong"
print("✅ P1 PASSED")

In [ ]:
# 💡 SOLUTION P1 — uncomment to reveal
# df["duration_sec"] = (df["ended_at"] - df["started_at"]).dt.total_seconds().astype(int)
# df["duration_min"] = round(df["duration_sec"] / 60, 2)
# df["hour_of_day"]  = df["started_at"].dt.hour
# df["day_of_week"]  = df["started_at"].dt.day_name()
# df["is_weekend"]   = df["started_at"].dt.dayofweek >= 5
# df["date"]         = df["started_at"].dt.date
# df["start_hour"]   = df["started_at"].dt.floor("h")

## P2 — Time-of-day buckets
Using `df` from P1, add column `time_of_day` based on `hour_of_day`:
- 0–5 → "Night"
- 6–9 → "Morning Rush"
- 10–15 → "Midday"
- 16–19 → "Evening Rush"
- 20–23 → "Night"

Then print the value_counts.

In [ ]:
# YOUR CODE HERE


print(df["time_of_day"].value_counts())

In [ ]:
# ✅ CHECK P2
assert df.loc[0, "time_of_day"] == "Morning Rush", f"got {df.loc[0,'time_of_day']}"
assert df.loc[4, "time_of_day"] == "Night",         f"got {df.loc[4,'time_of_day']}"
assert df.loc[3, "time_of_day"] == "Evening Rush",  f"got {df.loc[3,'time_of_day']}"
assert df.loc[2, "time_of_day"] == "Midday",        f"got {df.loc[2,'time_of_day']}"
print("✅ P2 PASSED")

In [ ]:
# 💡 SOLUTION P2
# conditions = [
#     df["hour_of_day"].between(0, 5),
#     df["hour_of_day"].between(6, 9),
#     df["hour_of_day"].between(10, 15),
#     df["hour_of_day"].between(16, 19),
#     df["hour_of_day"].between(20, 23),
# ]
# choices = ["Night", "Morning Rush", "Midday", "Evening Rush", "Night"]
# df["time_of_day"] = np.select(conditions, choices)

# ✂️ SECTION 2 — pd.cut() and pd.qcut()

## P3 — pd.cut() — fixed bins
Add column `trip_length` using `pd.cut()`:
- 0–600 sec → "Short"
- 601–1800 sec → "Medium"
- 1801+ sec → "Long"

Also add `duration_quartile` using `pd.qcut()` into 4 equal groups: Q1, Q2, Q3, Q4.

In [ ]:
# YOUR CODE HERE


print(df["trip_length"].value_counts())
print(df["duration_quartile"].value_counts())

In [ ]:
# ✅ CHECK P3
assert str(df.loc[0, "trip_length"]) == "Medium"   # 1200 sec
assert str(df.loc[1, "trip_length"]) == "Long"      # 2700 sec
assert df["duration_quartile"].nunique() == 4
assert set(df["duration_quartile"].dropna().unique()) == {"Q1","Q2","Q3","Q4"}
print("✅ P3 PASSED")

In [ ]:
# 💡 SOLUTION P3
# df["trip_length"] = pd.cut(
#     df["duration_sec"],
#     bins=[0, 600, 1800, float("inf")],
#     labels=["Short", "Medium", "Long"]
# )
# df["duration_quartile"] = pd.qcut(df["duration_sec"], q=4, labels=["Q1","Q2","Q3","Q4"])

# 📊 SECTION 3 — cumsum, expanding, rolling

## P4 — cumsum + expanding
Sort `df` by `started_at`. Then add:
- `cumulative_trips` — running trip count (1, 2, 3, ...)
- `cumulative_duration_sec` — running total of duration_sec
- `running_avg_duration` — expanding mean of duration_sec
- `running_max_duration` — expanding max of duration_sec

In [ ]:
df = df.sort_values("started_at").reset_index(drop=True)

# YOUR CODE HERE


df[["ride_id","started_at","duration_sec","cumulative_trips","cumulative_duration_sec","running_avg_duration","running_max_duration"]].head(5)

In [ ]:
# ✅ CHECK P4
assert df.loc[0, "cumulative_trips"] == 1
assert df.loc[2, "cumulative_trips"] == 3
assert df.loc[1, "cumulative_duration_sec"] == df.loc[0,"duration_sec"] + df.loc[1,"duration_sec"]
assert df.loc[0, "running_avg_duration"] == df.loc[0, "duration_sec"]
assert df.loc[0, "running_max_duration"] == df.loc[0, "duration_sec"]
print("✅ P4 PASSED")

In [ ]:
# 💡 SOLUTION P4
# df["cumulative_trips"]        = range(1, len(df) + 1)
# df["cumulative_duration_sec"] = df["duration_sec"].cumsum()
# df["running_avg_duration"]    = df["duration_sec"].expanding().mean()
# df["running_max_duration"]    = df["duration_sec"].expanding().max()

## P5 — rolling window
Add:
- `rolling_3_avg` — 3-trip rolling mean of duration_sec
- `rolling_3_max` — 3-trip rolling max
- `rolling_3_std` — 3-trip rolling std

Then find the `ride_id` of the **first trip** where `rolling_3_avg > 2000`.

In [ ]:
# YOUR CODE HERE


first_high = None  # YOUR CODE HERE

print(f"First trip with rolling_3_avg > 2000: {first_high}")
df[["ride_id","duration_sec","rolling_3_avg","rolling_3_max"]].head(6)

In [ ]:
# ✅ CHECK P5
assert pd.isna(df.loc[0, "rolling_3_avg"]), "row 0 should be NaN"
assert pd.isna(df.loc[1, "rolling_3_avg"]), "row 1 should be NaN"
assert not pd.isna(df.loc[2, "rolling_3_avg"]), "row 2 should have a value"
print("✅ P5 PASSED")

In [ ]:
# 💡 SOLUTION P5
# df["rolling_3_avg"] = df["duration_sec"].rolling(window=3).mean()
# df["rolling_3_max"] = df["duration_sec"].rolling(window=3).max()
# df["rolling_3_std"] = df["duration_sec"].rolling(window=3).std()
# above = df[df["rolling_3_avg"] > 2000]
# first_high = above.iloc[0]["ride_id"] if not above.empty else None

## P6 — Rolling per group (per station)
For each `start_station_id`, compute a **2-trip rolling average** of `duration_sec` within that station's trips (sorted by `started_at`).

Add column `station_rolling_2_avg`.

In [ ]:
# YOUR CODE HERE


df[["ride_id","start_station_id","started_at","duration_sec","station_rolling_2_avg"]].sort_values(["start_station_id","started_at"]).head(10)

In [ ]:
# ✅ CHECK P6
df_sorted = df.sort_values(["start_station_id","started_at"])
first_per_station = df_sorted.drop_duplicates(subset="start_station_id", keep="first")
assert first_per_station["station_rolling_2_avg"].isna().all(), "First trip per station should be NaN"
print("✅ P6 PASSED")

In [ ]:
# 💡 SOLUTION P6
# df = df.sort_values(["start_station_id","started_at"])
# df["station_rolling_2_avg"] = (
#     df.groupby("start_station_id")["duration_sec"]
#     .transform(lambda x: x.rolling(2).mean())
# )

# 🔀 SECTION 4 — melt

## P7 — melt (wide → long)
You have a station summary table. Melt it from wide to long format, then clean up column names.

**Wide:**
```
station | member_avg | casual_avg
```
**Long (target):**
```
station | user_type | avg_duration_sec
```
`user_type` should be `"member"` or `"casual"` (strip `_avg`).

In [ ]:
wide_df = (
    df.groupby(["start_station_id","member_casual"])["duration_sec"]
    .mean().round(0).reset_index()
    .pivot(index="start_station_id", columns="member_casual", values="duration_sec")
    .reset_index()
    .rename(columns={"start_station_id":"station","casual":"casual_avg","member":"member_avg"})
)
print("Wide format:")
print(wide_df)

long_df = None  # YOUR CODE HERE


print("\nLong format:")
print(long_df)

In [ ]:
# ✅ CHECK P7
assert long_df is not None
assert set(long_df.columns) == {"station","user_type","avg_duration_sec"}
assert set(long_df["user_type"].unique()) == {"member","casual"}
assert len(long_df) == len(wide_df) * 2
print("✅ P7 PASSED")

In [ ]:
# 💡 SOLUTION P7
# long_df = pd.melt(
#     wide_df,
#     id_vars=["station"],
#     value_vars=["member_avg","casual_avg"],
#     var_name="user_type",
#     value_name="avg_duration_sec"
# )
# long_df["user_type"] = long_df["user_type"].str.replace("_avg","")

# 🔧 SECTION 5 — pipe

## P8 — pipe chain
Build a full pipeline using `.pipe()` on `raw`. Each step is a separate function:

1. `add_duration(df)` — adds `duration_sec`
2. `add_time_features(df)` — adds `hour_of_day`, `day_of_week`, `is_weekend`, `date`
3. `add_trip_length(df)` — adds `trip_length` via `pd.cut()` (Short/Medium/Long)
4. `add_time_of_day(df)` — adds `time_of_day` via `np.select()` (Night/Morning Rush/Midday/Evening Rush)

Chain all 4 using `.pipe()`. Store result in `result`.

In [ ]:
def add_duration(df):
    # YOUR CODE HERE
    pass

def add_time_features(df):
    # YOUR CODE HERE
    pass

def add_trip_length(df):
    # YOUR CODE HERE
    pass

def add_time_of_day(df):
    # YOUR CODE HERE
    pass

result = raw.pipe(add_duration).pipe(add_time_features).pipe(add_trip_length).pipe(add_time_of_day)
result.head(3)

In [ ]:
# ✅ CHECK P8
required = {"duration_sec","hour_of_day","day_of_week","is_weekend","date","trip_length","time_of_day"}
assert required.issubset(result.columns), f"Missing: {required - set(result.columns)}"
assert result["trip_length"].isin(["Short","Medium","Long"]).all()
assert result["time_of_day"].isin(["Night","Morning Rush","Midday","Evening Rush"]).all()
print("✅ P8 PASSED")

In [ ]:
# 💡 SOLUTION P8
# def add_duration(df):
#     df = df.copy()
#     df["duration_sec"] = (df["ended_at"] - df["started_at"]).dt.total_seconds().astype(int)
#     return df
#
# def add_time_features(df):
#     df = df.copy()
#     df["hour_of_day"] = df["started_at"].dt.hour
#     df["day_of_week"] = df["started_at"].dt.day_name()
#     df["is_weekend"]  = df["started_at"].dt.dayofweek >= 5
#     df["date"]        = df["started_at"].dt.date
#     return df
#
# def add_trip_length(df):
#     df = df.copy()
#     df["trip_length"] = pd.cut(df["duration_sec"], bins=[0,600,1800,float("inf")], labels=["Short","Medium","Long"])
#     return df
#
# def add_time_of_day(df):
#     df = df.copy()
#     conds = [df["hour_of_day"].between(0,5), df["hour_of_day"].between(6,9),
#              df["hour_of_day"].between(10,15), df["hour_of_day"].between(16,19), df["hour_of_day"].between(20,23)]
#     choices = ["Night","Morning Rush","Midday","Evening Rush","Night"]
#     df["time_of_day"] = np.select(conds, choices)
#     return df
#
# result = raw.pipe(add_duration).pipe(add_time_features).pipe(add_trip_length).pipe(add_time_of_day)

# 🏁 SECTION 6 — Full combo challenge

## P9 — Daily station summary
Build a daily summary per start station. For each `(date, start_station_id)`:
- `trip_count`
- `avg_duration_min` (rounded to 1 dp)
- `pct_member` — % of trips by members (rounded to 1 dp)
- `busiest_hour` — hour with most departures that day from that station
- `cumulative_trips` — running total of trips per station over dates

Join with `stations` to add `station_name` and `region`.
Store as `daily_summary`.

In [ ]:
daily_summary = None  # YOUR CODE HERE


daily_summary

In [ ]:
# ✅ CHECK P9
assert daily_summary is not None
required = {"date","start_station_id","station_name","region","trip_count",
            "avg_duration_min","pct_member","busiest_hour","cumulative_trips"}
assert required.issubset(daily_summary.columns), f"Missing: {required - set(daily_summary.columns)}"
assert (daily_summary["trip_count"] > 0).all()
print("✅ P9 PASSED")

In [ ]:
# 💡 SOLUTION P9
# base = df.copy()
# base["date"]       = base["started_at"].dt.date
# base["hour"]       = base["started_at"].dt.hour
# base["duration_min"] = base["duration_sec"] / 60
#
# busiest = (
#     base.groupby(["date","start_station_id","hour"]).size().reset_index(name="cnt")
#     .sort_values("cnt", ascending=False)
#     .drop_duplicates(subset=["date","start_station_id"])
#     .rename(columns={"hour":"busiest_hour"})[["date","start_station_id","busiest_hour"]]
# )
#
# daily_summary = (
#     base.groupby(["date","start_station_id"]).agg(
#         trip_count      =("ride_id","count"),
#         avg_duration_min=("duration_min", lambda x: round(x.mean(),1)),
#         pct_member      =("member_casual", lambda x: round(100*(x=="member").sum()/len(x),1)),
#     ).reset_index()
# )
# daily_summary = pd.merge(daily_summary, busiest, on=["date","start_station_id"])
# daily_summary = pd.merge(daily_summary, stations[["station_id","station_name","region"]],
#                          left_on="start_station_id", right_on="station_id").drop(columns="station_id")
# daily_summary = daily_summary.sort_values(["start_station_id","date"])
# daily_summary["cumulative_trips"] = daily_summary.groupby("start_station_id")["trip_count"].cumsum()

# 🌐 SECTION 7 — Load Real Datasets from Online Sources
All datasets load directly from their official S3 buckets — no download needed.
Each cell is independent. Run whichever dataset you want to practice on.

## Dataset Index — Direct S3 Links

| System | City | S3 Bucket | File Pattern |
|---|---|---|---|
| Divvy | Chicago | divvy-tripdata.s3.amazonaws.com | `YYYYMM-divvy-tripdata.zip` |
| Citi Bike | NYC | s3.amazonaws.com/tripdata | `YYYYMM-citibike-tripdata.csv.zip` |
| Capital Bikeshare | DC | s3.amazonaws.com/capitalbikeshare-data | `YYYYMM-capitalbikeshare-tripdata.zip` |
| Bay Wheels | SF | baywheels-data.s3.amazonaws.com | `YYYYMM-baywheels-tripdata.zip` |
| Bluebikes | Boston | s3.amazonaws.com/hubway-data | `YYYYMM-bluebikes-tripdata.zip` |

**Schema type:**
- Divvy, Citi Bike, Bay Wheels → same Lyft schema (`started_at`, `ended_at`, `member_casual`)
- Capital Bikeshare → `Duration`, `Start date`, `Member type`
- Bluebikes → `tripduration`, `starttime`, `usertype`, `birth year`, `gender`

## 1. Divvy — Chicago (Lyft schema)
S3 index: https://divvy-tripdata.s3.amazonaws.com/index.html

In [ ]:
# Change MONTH to any YYYYMM value available in the S3 index
MONTH = "202401"

url = f"https://divvy-tripdata.s3.amazonaws.com/{MONTH}-divvy-tripdata.zip"
print(f"Loading: {url}")

divvy = pd.read_csv(url, compression="zip")
divvy["started_at"] = pd.to_datetime(divvy["started_at"])
divvy["ended_at"]   = pd.to_datetime(divvy["ended_at"])
divvy["duration_sec"] = (divvy["ended_at"] - divvy["started_at"]).dt.total_seconds()

print(f"Shape: {divvy.shape}")
print(f"Columns: {divvy.columns.tolist()}")
print(f"Null counts:\n{divvy.isnull().sum()}")
divvy.head(3)

## 2. Citi Bike — NYC (Lyft schema)
S3 index: https://s3.amazonaws.com/tripdata/index.html

In [ ]:
MONTH = "202401"

url = f"https://s3.amazonaws.com/tripdata/{MONTH}-citibike-tripdata.csv.zip"
print(f"Loading: {url}")

citibike = pd.read_csv(url, compression="zip")
citibike["started_at"] = pd.to_datetime(citibike["started_at"])
citibike["ended_at"]   = pd.to_datetime(citibike["ended_at"])
citibike["duration_sec"] = (citibike["ended_at"] - citibike["started_at"]).dt.total_seconds()

print(f"Shape: {citibike.shape}")
print(f"Columns: {citibike.columns.tolist()}")
print(f"member_casual counts:\n{citibike['member_casual'].value_counts()}")
print(f"rideable_type counts:\n{citibike['rideable_type'].value_counts()}")
citibike.head(3)

## 3. Capital Bikeshare — DC (old schema: Duration, Start date, Member type)
S3 index: https://s3.amazonaws.com/capitalbikeshare-data/index.html

In [ ]:
MONTH = "202401"

url = f"https://s3.amazonaws.com/capitalbikeshare-data/{MONTH}-capitalbikeshare-tripdata.zip"
print(f"Loading: {url}")

capital = pd.read_csv(url, compression="zip")

# Normalize column names (Capital uses spaces and title case)
capital.columns = capital.columns.str.strip().str.lower().str.replace(" ", "_")
capital["started_at"] = pd.to_datetime(capital["started_at"])
capital["ended_at"]   = pd.to_datetime(capital["ended_at"])
capital["duration_sec"] = (capital["ended_at"] - capital["started_at"]).dt.total_seconds()

print(f"Shape: {capital.shape}")
print(f"Columns: {capital.columns.tolist()}")
print(f"member_type counts:\n{capital['member_type'].value_counts()}")
capital.head(3)

## 4. Bay Wheels — San Francisco (Lyft schema)
S3 index: https://baywheels-data.s3.amazonaws.com/index.html

In [ ]:
MONTH = "202401"

url = f"https://baywheels-data.s3.amazonaws.com/{MONTH}-baywheels-tripdata.zip"
print(f"Loading: {url}")

baywheels = pd.read_csv(url, compression="zip")
baywheels["started_at"] = pd.to_datetime(baywheels["started_at"])
baywheels["ended_at"]   = pd.to_datetime(baywheels["ended_at"])
baywheels["duration_sec"] = (baywheels["ended_at"] - baywheels["started_at"]).dt.total_seconds()

print(f"Shape: {baywheels.shape}")
print(f"Columns: {baywheels.columns.tolist()}")
print(f"member_casual counts:\n{baywheels['member_casual'].value_counts()}")
baywheels.head(3)

## 5. Bluebikes — Boston (old schema: tripduration, usertype, birth year, gender)
S3 index: https://s3.amazonaws.com/hubway-data/index.html

In [ ]:
MONTH = "202401"

url = f"https://s3.amazonaws.com/hubway-data/{MONTH}-bluebikes-tripdata.zip"
print(f"Loading: {url}")

bluebikes = pd.read_csv(url, compression="zip")
bluebikes.columns = bluebikes.columns.str.strip().str.lower().str.replace(" ", "_")
bluebikes["starttime"] = pd.to_datetime(bluebikes["starttime"])
bluebikes["stoptime"]  = pd.to_datetime(bluebikes["stoptime"])

print(f"Shape: {bluebikes.shape}")
print(f"Columns: {bluebikes.columns.tolist()}")
print(f"usertype counts:\n{bluebikes['usertype'].value_counts()}")
print(f"gender counts:\n{bluebikes['gender'].value_counts()}")
bluebikes.head(3)

## 6. Available months across all datasets

Run this to see all available files in each S3 bucket — helps you pick which month to load.

In [ ]:
import urllib.request
import xml.etree.ElementTree as ET

def list_s3_bucket(bucket_url, keyword=""):
    """List all files in a public S3 bucket, optionally filtered by keyword."""
    try:
        with urllib.request.urlopen(bucket_url) as r:
            tree = ET.parse(r)
        ns = "{http://s3.amazonaws.com/doc/2006-03-01/}"
        keys = [el.text for el in tree.iter(f"{ns}Key") if keyword in el.text]
        return sorted(keys)[-12:]  # last 12 files
    except Exception as e:
        return [f"Error: {e}"]

print("=== DIVVY (Chicago) — last 12 files ===")
for f in list_s3_bucket("https://divvy-tripdata.s3.amazonaws.com/", "divvy-tripdata.zip"):
    print(f"  https://divvy-tripdata.s3.amazonaws.com/{f}")

print("\n=== CITI BIKE (NYC) — last 12 files ===")
for f in list_s3_bucket("https://s3.amazonaws.com/tripdata/", "citibike-tripdata"):
    print(f"  https://s3.amazonaws.com/tripdata/{f}")

print("\n=== CAPITAL BIKESHARE (DC) — last 12 files ===")
for f in list_s3_bucket("https://s3.amazonaws.com/capitalbikeshare-data/", "tripdata.zip"):
    print(f"  https://s3.amazonaws.com/capitalbikeshare-data/{f}")

print("\n=== BAY WHEELS (SF) — last 12 files ===")
for f in list_s3_bucket("https://baywheels-data.s3.amazonaws.com/", "tripdata.zip"):
    print(f"  https://baywheels-data.s3.amazonaws.com/{f}")

print("\n=== BLUEBIKES (Boston) — last 12 files ===")
for f in list_s3_bucket("https://s3.amazonaws.com/hubway-data/", "bluebikes-tripdata.zip"):
    print(f"  https://s3.amazonaws.com/hubway-data/{f}")